# HAR Own Dataset Training

Use this notebook in Google Colab. Upload `har_colab_training_bundle.zip`, train the 200x27 model, then download the trained model zip and copy it back to the Raspberry Pi.

## 1. Upload Bundle

In [ ]:
from google.colab import files

uploaded = files.upload()
bundle_name = next(iter(uploaded))
print('Uploaded:', bundle_name)

## 2. Unpack And Check Dataset

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

workdir = Path('/content/har_training')
if workdir.exists():
    shutil.rmtree(workdir)
workdir.mkdir(parents=True)

with zipfile.ZipFile(bundle_name, 'r') as archive:
    archive.extractall(workdir)

os.chdir(workdir)
print('Working directory:', Path.cwd())
!ls -lah
!find datasets/raw_sessions -maxdepth 1 -type d -name 'session_*' | wc -l

## 3. Install/Import Dependencies

In [ ]:
!pip install -q joblib scikit-learn

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 4. Build 200x27 Windows

This uses a session split so every participant can help the demo model, while some sessions are still held out for validation/test.

In [ ]:
!python build_window_dataset.py --dataset-root datasets --split-by session --stride-seconds 3.0 --max-source-gap-seconds 1.5

import json
summary = json.load(open('datasets/processed/dataset_summary.json'))
print('Total windows:', summary['total_windows'])
print('Class counts:', summary['class_counts'])
print('Participants:', summary['participants'])
print('Split:', summary['split']['split_by'], {k: len(v) for k, v in summary['split']['splits'].items()})

## 5. Train Model

In [ ]:
!python train_own_har_model.py \
  --windows-path datasets/processed/windows_200x27.jsonl \
  --split-dir datasets/splits \
  --model-dir model \
  --epochs 80 \
  --batch-size 64 \
  --learning-rate 0.0005 \
  --patience 12

## 6. Evaluate Held-Out Sessions

In [ ]:
!python evaluate_model.py \
  --windows-path datasets/processed/windows_200x27.jsonl \
  --split-dir datasets/splits \
  --model-path model/own_har_200x27_model.keras \
  --scaler-path model/own_har_200x27_scaler.pkl \
  --label-encoder-path model/own_har_200x27_label_encoder.pkl \
  --output-path model/own_har_200x27_evaluation.json \
  --split test

import json
import numpy as np
report = json.load(open('model/own_har_200x27_evaluation.json'))
print('Accuracy:', report['accuracy'])
print('Labels:', report['labels'])
print(np.asarray(report['confusion_matrix']))

## 7. Download Model Files

In [ ]:
!zip -j own_har_200x27_colab_model.zip \
  model/own_har_200x27_model.keras \
  model/own_har_200x27_scaler.pkl \
  model/own_har_200x27_label_encoder.pkl \
  model/own_har_200x27_metrics.json \
  model/own_har_200x27_evaluation.json \
  datasets/processed/dataset_summary.json \
  datasets/splits/split_summary.json

files.download('own_har_200x27_colab_model.zip')